In [1]:
import os, glob, random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import wandb
 
# Find the competition folder automatically
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
print("Data folder:", DATA_DIR)
 
H, W = 128, 800          # every image is 800 wide x 128 tall
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
EPOCHS = 8
BATCH_SIZE = 16
LR = 1e-3
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
 
# Fix random seeds so results can be repeated
random.seed(42); np.random.seed(42); torch.manual_seed(42)

Data folder: /kaggle/input/competitions/industrial-surface-defect-detection/public


In [2]:
# Masks are column-major (go down each column first), pixel numbers start at 1
 
def rle_to_mask(rle):
    mask = np.zeros(H * W, dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        nums = list(map(int, rle.split()))
        for start, length in zip(nums[0::2], nums[1::2]):
            mask[start - 1 : start - 1 + length] = 1
    return mask.reshape(W, H).T          # column-major -> (H, W)
 
def mask_to_rle(mask):
    pixels = np.concatenate([[0], mask.T.flatten(), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))

In [3]:
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
 
# One row per image, with its 4 RLE strings in class order
rles = train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")[CLASSES]
image_ids = rles.index.tolist()
 
# Simple 90% train / 10% validation split
random.shuffle(image_ids)
n_val = len(image_ids) // 10
val_ids, train_ids = image_ids[:n_val], image_ids[n_val:]
print("Train:", len(train_ids), " Val:", len(val_ids))
 
class DefectDataset(Dataset):
    def __init__(self, ids, folder, train=True, with_masks=True):
        self.ids, self.folder = ids, folder
        self.train, self.with_masks = train, with_masks
 
    def __len__(self):
        return len(self.ids)
 
    def __getitem__(self, i):
        img_id = self.ids[i]
        # Read as grayscale (1 channel) and scale to 0-1
        img = cv2.imread(f"{self.folder}/{img_id}.jpg", cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32) / 255.0
 
        if not self.with_masks:                      # test images
            return torch.tensor(img)[None], img_id
 
        mask = np.stack([rle_to_mask(r) for r in rles.loc[img_id]])   # (4, H, W)
 
        # Simple augmentation: random horizontal / vertical flip
        if self.train and random.random() < 0.5:
            img, mask = img[:, ::-1], mask[:, :, ::-1]
        if self.train and random.random() < 0.5:
            img, mask = img[::-1, :], mask[:, ::-1, :]
 
        img = torch.tensor(img.copy())[None]         # (1, H, W)
        mask = torch.tensor(mask.copy()).float()     # (4, H, W)
        return img, mask
 
train_loader = DataLoader(DefectDataset(train_ids, f"{DATA_DIR}/train", train=True),
                          batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(DefectDataset(val_ids, f"{DATA_DIR}/train", train=False),
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
 

Train: 10254  Val: 1139


In [4]:
# Two 3x3 convolutions, each followed by BatchNorm and ReLU
class DoubleConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)
 
class ScratchUNet(nn.Module):
    def __init__(self, in_ch=1, n_classes=4, base=16):
        super().__init__()
        # Encoder: image gets smaller, channels get bigger
        self.enc1 = DoubleConv(in_ch, base)
        self.enc2 = DoubleConv(base, base * 2)
        self.enc3 = DoubleConv(base * 2, base * 4)
        self.enc4 = DoubleConv(base * 4, base * 8)
        self.bottleneck = DoubleConv(base * 8, base * 16)
        self.pool = nn.MaxPool2d(2)
 
        # Decoder: upsample and join with the matching encoder output (skip connection)
        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = DoubleConv(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = DoubleConv(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
 
        # One output channel per defect class
        self.out = nn.Conv2d(base, n_classes, 1)
 
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
 
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)              # raw scores (logits), shape (B, 4, H, W)
 
model = ScratchUNet().to(DEVICE)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Parameters: 1943812


In [5]:
# Loss = BCE (per pixel) + Dice (overlap)
def dice_loss(logits, target, eps=1.0):
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1 - ((2 * inter + eps) / (union + eps)).mean()
 
def loss_fn(logits, target):
    return F.binary_cross_entropy_with_logits(logits, target) + dice_loss(logits, target)
 
# Dice score per (image, class); if both prediction and truth are empty, score = 1
def dice_score(pred, target):
    inter = (pred * target).sum(dim=(2, 3))
    total = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = torch.where(total == 0, torch.ones_like(total), 2 * inter / total.clamp(min=1))
    return dice.mean().item()
 
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
 
# Log in to W&B (add your key in Add-ons > Secrets with the name WANDB_API_KEY)
from kaggle_secrets import UserSecretsClient
wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
wandb.init(project="24f3005134-t32026", name="unet-from-scratch",
           config={"model": "UNet from scratch", "base_channels": 16, "epochs": EPOCHS,
                   "batch_size": BATCH_SIZE, "lr": LR, "loss": "BCE + Dice", "input": "grayscale"})
 
best_dice = 0
for epoch in range(EPOCHS):
    # train 
    model.train()
    train_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(imgs), masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    scheduler.step()
 
    # validate
    model.eval()
    val_loss, val_dice = 0, 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            val_loss += loss_fn(logits, masks).item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_dice += dice_score(preds, masks)
    val_loss /= len(val_loader)
    val_dice /= len(val_loader)
 
    print(f"Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_dice={val_dice:.4f}")
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": val_loss, "val_dice": val_dice})
 
    # Keep the best model
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "/kaggle/working/unet_scratch_best.pth")
 
print("Best val dice:", best_dice)
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261110_135502-f5f74zwb
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run unet-from-scratch
wandb: ⭐️ View project at https://wandb.ai/24f3005134-dl-genai-project/24f3005134-t32026
wandb: 🚀 View run at https://wandb.ai/24f3005134-dl-genai-project/24f3005134-t32026/runs/f5f74zwb


Epoch 1/8  train_loss=1.1309  val_loss=1.0183  val_dice=0.7483
Epoch 2/8  train_loss=1.0016  val_loss=0.9944  val_dice=0.6818
Epoch 3/8  train_loss=0.7214  val_loss=0.3485  val_dice=0.7738
Epoch 4/8  train_loss=0.3258  val_loss=0.2533  val_dice=0.8533
Epoch 5/8  train_loss=0.2590  val_loss=0.2445  val_dice=0.8518
Epoch 6/8  train_loss=0.2496  val_loss=0.2296  val_dice=0.8578
Epoch 7/8  train_loss=0.2402  val_loss=0.2262  val_dice=0.8606


wandb: updating run metadata


Epoch 8/8  train_loss=0.2367  val_loss=0.2196  val_dice=0.8598
Best val dice: 0.8605776247051027


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:      epoch ▁▂▃▄▅▆▇█
wandb: train_loss █▇▅▂▁▁▁▁
wandb:   val_dice ▄▁▅█████
wandb:   val_loss ██▂▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:      epoch 8
wandb: train_loss 0.23667
wandb:   val_dice 0.85984
wandb:   val_loss 0.21962
wandb: 
wandb: 🚀 View run unet-from-scratch at: https://wandb.ai/24f3005134-dl-genai-project/24f3005134-t32026/runs/f5f74zwb
wandb: ⭐️ View project at: https://wandb.ai/24f3005134-dl-genai-project/24f3005134-t32026
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261110_135502-f5f74zwb/logs


In [6]:
model.load_state_dict(torch.load("/kaggle/working/unet_scratch_best.pth", map_location=DEVICE))
model.eval()
 
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
test_ids = sub["image_id"].unique().tolist()
test_loader = DataLoader(DefectDataset(test_ids, f"{DATA_DIR}/test", with_masks=False),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
 
MIN_AREA = 300                       # ignore tiny predicted regions (likely noise)
predictions = {}
with torch.no_grad():
    for imgs, ids in test_loader:
        probs = torch.sigmoid(model(imgs.to(DEVICE))).cpu().numpy()
        for img_id, p in zip(ids, probs):
            for c, cls in enumerate(CLASSES):
                m = (p[c] > 0.5).astype(np.uint8)
                predictions[(img_id, cls)] = mask_to_rle(m) if m.sum() >= MIN_AREA else " "
 
sub["mask_rle"] = [predictions[(i, d)] for i, d in zip(sub["image_id"], sub["defect_type"])]
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.head())
print("Non-empty predictions:", (sub["mask_rle"] != " ").sum(), "of", len(sub))

   ID    image_id defect_type mask_rle
0   1  WAFNKCRR7Q    defect_a         
1   2  WAFNKCRR7Q    defect_b         
2   3  WAFNKCRR7Q    defect_c         
3   4  WAFNKCRR7Q    defect_d         
4   5  YZXKEWSUDD    defect_a         
Non-empty predictions: 98 of 12000
